# Task 7 — Hypothesis testing

**Problem-statement step — test the following hypotheses:**
1. Older individuals lean toward traditional in-store shopping.
2. Customers with children prefer the convenience of online shopping.
3. Physical-store sales may be cannibalised by other channels.
4. Does the US significantly outperform the rest of the world in total purchases?

All tests use α = 0.05.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 120)
sns.set_theme(style="whitegrid")

# Locate the raw data whether the notebook is opened from its own folder
# (notebooks/) or from the project root.
import os
_CANDIDATES = ["marketing_data.csv", "../marketing_data.csv",
               os.path.join(os.path.dirname(os.getcwd()), "marketing_data.csv")]
DATA_PATH = next((p for p in _CANDIDATES if os.path.exists(p)), "marketing_data.csv")
print("Using data file:", DATA_PATH)

Using data file: ../marketing_data.csv


In [2]:
REFERENCE_YEAR = 2015  # data compiled just after the last enrolment (Jun 2014)
SPEND_COLS = ["MntWines", "MntFruits", "MntMeatProducts",
              "MntFishProducts", "MntSweetProducts", "MntGoldProds"]
CHANNEL_COLS = ["NumWebPurchases", "NumCatalogPurchases", "NumStorePurchases"]

def load_and_prepare(path=DATA_PATH, engineer=True):
    """Load -> fix dtypes -> clean categories -> impute income -> features."""
    df = pd.read_csv(path)
    df.columns = df.columns.str.strip()                     # ' Income ' -> 'Income'
    if not pd.api.types.is_numeric_dtype(df["Income"]):     # '$84,835.00' -> float
        df["Income"] = (df["Income"].astype("string")
                        .str.replace(r"[\$,]", "", regex=True).str.strip().astype(float))
    df["Dt_Customer"] = pd.to_datetime(df["Dt_Customer"], format="%m/%d/%y")
    df["Marital_Status"] = df["Marital_Status"].replace(
        {"Alone": "Single", "YOLO": "Single", "Absurd": "Single"})
    df["Income"] = df.groupby(["Education", "Marital_Status"])["Income"].transform(
        lambda s: s.fillna(s.median()))
    df["Income"] = df["Income"].fillna(df["Income"].median())
    if engineer:
        df["Kids"] = df["Kidhome"] + df["Teenhome"]
        df["Age"] = REFERENCE_YEAR - df["Year_Birth"]
        df["Total_Spending"] = df[SPEND_COLS].sum(axis=1)
        df["Total_Purchases"] = df[CHANNEL_COLS].sum(axis=1)
        df["Total_Accepted_Cmp"] = df[["AcceptedCmp1", "AcceptedCmp2", "AcceptedCmp3",
                                       "AcceptedCmp4", "AcceptedCmp5"]].sum(axis=1)
        df["Has_Child"] = (df["Kids"] > 0).astype(int)
        df["Is_US"] = (df["Country"] == "US").astype(int)
    return df

def iqr_bounds(s, k=1.5):
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    return q1 - k * (q3 - q1), q3 + k * (q3 - q1)

def treat_outliers(df):
    """Drop impossible ages and winsorise heavy-tailed continuous columns."""
    df = df[df["Age"] <= 100].copy()
    for col in ["Income", "Total_Spending", "Total_Purchases", "NumWebVisitsMonth"]:
        lo, hi = iqr_bounds(df[col])
        df[col] = df[col].clip(lo, hi)
    return df

In [3]:
from scipy import stats
ALPHA = 0.05
def verdict(p): return "REJECT H0 (significant)" if p < ALPHA else "FAIL TO REJECT H0 (n.s.)"
df = treat_outliers(load_and_prepare())

## H1 — Older customers prefer in-store shopping
H0: no relationship between `Age` and `NumStorePurchases`. H1: a positive association.

In [4]:
r_p, p_p = stats.pearsonr(df["Age"], df["NumStorePurchases"])
r_s, p_s = stats.spearmanr(df["Age"], df["NumStorePurchases"])
print(f"Pearson  r = {r_p:+.3f}  (p = {p_p:.2e})")
print(f"Spearman r = {r_s:+.3f}  (p = {p_s:.2e})")
print("Verdict:", verdict(p_p))

Pearson  r = +0.139  (p = 3.47e-11)
Spearman r = +0.171  (p = 4.19e-16)
Verdict: REJECT H0 (significant)


**Reading:** a *weak but statistically significant* positive correlation — older customers do buy somewhat more in store, though the effect is small.

## H2 — Customers with children shop more online
Childless customers simply buy more of *everything*, so we compare the **web share** of purchases (not the raw count).
H0: equal web share. H1: customers with children have a larger web share.

In [5]:
total_ch = df[CHANNEL_COLS].sum(axis=1)
web_share = (df["NumWebPurchases"] / total_ch).astype(float)
valid = total_ch > 0
with_kids = web_share[valid & (df["Has_Child"] == 1)].dropna()
no_kids   = web_share[valid & (df["Has_Child"] == 0)].dropna()
u, p = stats.mannwhitneyu(with_kids, no_kids, alternative="greater")
print(f"Mean web share - with children : {with_kids.mean():.1%}")
print(f"Mean web share - no children   : {no_kids.mean():.1%}")
print(f"Mann-Whitney U = {u:.0f}  (p = {p:.2e})")
print("Verdict:", verdict(p))

Mean web share - with children : 35.3%
Mean web share - no children   : 27.3%
Mann-Whitney U = 714298  (p = 1.01e-52)
Verdict: REJECT H0 (significant)


**Reading:** customers with children devote a significantly **larger share** of purchases to the web channel — the convenience hypothesis is *supported*.

## H3 — Physical-store sales cannibalised by other channels
H0: store purchases not negatively correlated with other channels. H1: negative correlation (substitution).

In [6]:
for other in ["NumWebPurchases", "NumCatalogPurchases"]:
    r, p = stats.pearsonr(df["NumStorePurchases"], df[other])
    print(f"Store vs {other:20s}: r = {r:+.3f}  (p = {p:.2e})")

Store vs NumWebPurchases     : r = +0.502  (p = 2.68e-143)
Store vs NumCatalogPurchases : r = +0.519  (p = 1.90e-154)


**Reading:** correlations are **positive**, not negative — heavy buyers purchase more through *every* channel, so channels are complements, not substitutes. No cannibalisation.

## H4 — Does the US outperform the rest of the world?
H0: US total purchases equal the rest of the world. H1: they differ.

In [7]:
us  = df.loc[df["Country"] == "US", "Total_Purchases"]
row = df.loc[df["Country"] != "US", "Total_Purchases"]
u, p = stats.mannwhitneyu(us, row, alternative="two-sided")
print(f"US (n={len(us)})   mean total purchases : {us.mean():.2f}")
print(f"Rest (n={len(row)}) mean total purchases : {row.mean():.2f}")
print(f"Mann-Whitney U = {u:.0f}  (p = {p:.3f})")
print("Verdict:", verdict(p))

US (n=109)   mean total purchases : 13.51
Rest (n=2128) mean total purchases : 12.49
Mann-Whitney U = 125544  (p = 0.145)
Verdict: FAIL TO REJECT H0 (n.s.)


**Reading:** the US does **not** significantly outperform the rest of the world — purchase volumes are statistically comparable.

### Summary
| # | Hypothesis | Result |
|---|------------|--------|
| H1 | Older → in-store | Weak support (small but significant +corr) |
| H2 | Children → online | **Supported** (larger web share) |
| H3 | Store cannibalisation | Not supported (channels are complements) |
| H4 | US > rest of world | Not supported (no significant difference) |